In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, current_date, input_file_name, lit
import datetime

STREAM_NAME   = "energy_usage"
BUCKET        = "s3://energy-pipline-prod"
RAW_PATH      = f"{BUCKET}/raw/energy_usage_stream_v2.csv"
BRONZE_PATH   = f"{BUCKET}/bronze/energy_usage/"         # ← updated
CATALOG_TABLE = "energy_catalog.raw.bronze_energy_usage"
BAD_RECORDS   = f"{BUCKET}/bronze/logs/bad_records/energy_usage/"  # ← updated
batch_date    = datetime.date.today().strftime("%Y-%m-%d")

print(f"Source : {RAW_PATH}")
print(f"Target : {BRONZE_PATH}")
print(f"Date   : {batch_date}")

In [0]:
energy_schema = StructType([
    StructField("household_id",          StringType(),  True),
    StructField("region_name",           StringType(),  True),
    StructField("city_name",             StringType(),  True),
    StructField("meter_type",            StringType(),  True),
    StructField("customer_category",     StringType(),  True),
    StructField("grid_zone",             StringType(),  True),
    StructField("voltage_reading",       DoubleType(),  True),
    StructField("current_reading",       DoubleType(),  True),
    StructField("active_power_kw",       DoubleType(),  True),
    StructField("reactive_power_kvar",   DoubleType(),  True),
    StructField("energy_usage_kwh",      DoubleType(),  True),
    StructField("frequency_hz",          DoubleType(),  True),
    StructField("load_factor",           DoubleType(),  True),
    StructField("peak_demand_kw",        DoubleType(),  True),
    StructField("offpeak_demand_kw",     DoubleType(),  True),
    StructField("daily_consumption_kwh", DoubleType(),  True),
    StructField("timestamp",             StringType(),  True),
])
print(f"Schema ready: {len(energy_schema.fields)} columns")

In [0]:
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("badRecordsPath", BAD_RECORDS)
    .schema(energy_schema)
    .load(RAW_PATH)
)
print(f"Rows read: {df_raw.count()}")   # expect 50,000
df_raw.show(3)

In [0]:
from pyspark.sql.functions import col

df_bronze = (
    df_raw
    .withColumn("_batch_date",   lit(batch_date))
    .withColumn("_ingestion_ts", current_timestamp())
    .withColumn("_source_file",  col("_metadata.file_path"))  # ← fixed
    .withColumn("_stream_name",  lit(STREAM_NAME))
)
print(f"Total columns: {len(df_bronze.columns)}")  # expect 21

In [0]:
(df_bronze.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("_batch_date")
    .save(BRONZE_PATH)
)
print("Write complete")
print(f"Rows in Delta: {spark.read.format('delta').load(BRONZE_PATH).count()}")

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG_TABLE}
    USING DELTA
    LOCATION '{BRONZE_PATH}'
""")
spark.sql(f"SELECT COUNT(*) AS total FROM {CATALOG_TABLE}").show()

In [0]:
spark.sql(f"OPTIMIZE {CATALOG_TABLE} ZORDER BY (household_id)")
print("Done. Bronze energy_usage complete.") 